In [ ]:
import os, sys, json, time, random, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ijson", "tokenizers", "tqdm"], check=False)

import numpy as np, torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print("gpu count:", torch.cuda.device_count())

In [ ]:
import glob
hits = glob.glob("/kaggle/input/**/bpe32k_train.bin", recursive=True)
print(hits)

In [ ]:
import os, glob, json
from pathlib import Path
import numpy as np
from tokenizers import Tokenizer

DATA = Path("/kaggle/input/datasets/megs132/capstone-1")

for p in sorted(DATA.iterdir()):
    print(f"{p.stat().st_size/1e6:9.1f} MB  {p.name}")

WORK     = "/kaggle/working"
CKPT_DIR = f"{WORK}/ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

TRAIN_BIN, VAL_BIN = DATA / "bpe32k_train.bin", DATA / "bpe32k_val.bin"
META_PATH, TOK_PATH = DATA / "bpe32k_meta.json", DATA / "bpe32k.json"
TEST_PATH = DATA / "test.json"
VAL_PROMPTS_PATH, VAL_REFS_PATH = DATA / "val_prompts.json", DATA / "val_refs.json"

for p in (TRAIN_BIN, VAL_BIN, META_PATH, TOK_PATH, TEST_PATH,
          VAL_PROMPTS_PATH, VAL_REFS_PATH):
    assert p.exists(), f"missing: {p}"

META = json.load(open(META_PATH))
VOCAB_SIZE = META["vocab_size"]
PAD_ID = META["special_token_ids"]["pad"]
EOS_ID = META["special_token_ids"]["eos"]

tok = Tokenizer.from_file(str(TOK_PATH))
train_bin = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_bin   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")

test_data   = json.load(open(TEST_PATH))
VAL_PROMPTS = json.load(open(VAL_PROMPTS_PATH))
VAL_REFS    = json.load(open(VAL_REFS_PATH))

print(f"\nvocab {VOCAB_SIZE:,} | pad {PAD_ID} | eos {EOS_ID}")
print(f"train {len(train_bin):,} tokens | val {len(val_bin):,} tokens")
print(f"test {len(test_data)} | val prompts {len(VAL_PROMPTS)}")
print("\nsample:", tok.decode(train_bin[:60].astype(np.int64).tolist())[:300])

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x = x.float()                                   # fp32 reduction
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x * rms).to(dtype) * self.weight


class SwiGLU(nn.Module):
    def __init__(self, dim, mult=4, multiple_of=64):
        super().__init__()
        hidden = int(dim * mult * 2 / 3)
        hidden = multiple_of * ((hidden + multiple_of - 1) // multiple_of)

        self.w1 = nn.Linear(dim, hidden, bias=False)    # gate branch
        self.w2 = nn.Linear(dim, hidden, bias=False)    # value branch
        self.w3 = nn.Linear(hidden, dim, bias=False)    # project back

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


# ---------------- tests ----------------

torch.manual_seed(42)
B, T, D = 4, 32, 768
x = torch.randn(B, T, D)

norm = RMSNorm(D)
y = norm(x)
rms = y.pow(2).mean(-1).sqrt()
assert y.shape == x.shape
assert torch.allclose(rms, torch.ones_like(rms), atol=1e-4)
print(f"RMSNorm   ok | out RMS {rms.mean():.6f} | params {sum(p.numel() for p in norm.parameters()):,}")

# fp16 stability: large activations must not overflow
xb = torch.randn(2, 8, D) * 300
assert torch.isfinite(norm(xb.half() if torch.cuda.is_available() else xb)).all()
print("RMSNorm   fp16 overflow test ok")

mlp = SwiGLU(D)
z = mlp(x)
assert z.shape == x.shape
print(f"SwiGLU    ok | hidden {mlp.w1.out_features} | params {sum(p.numel() for p in mlp.parameters()):,}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RoPE(nn.Module):
    """Contiguous-half rotary embeddings.
    x = [x1 | x2] -> [x1*cos - x2*sin, x2*cos + x1*sin]
    This convention MUST match the inference path exactly."""

    def __init__(self, head_dim, max_seq):
        super().__init__()
        assert head_dim % 2 == 0, "head_dim must be even for RoPE"
        self.head_dim, self.max_seq = head_dim, max_seq

        inv_freq = 1.0 / (10000 ** (
            torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_seq, dtype=torch.float32), inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, x, pos0=0):
        """x: (B, H, T, head_dim). pos0 is the absolute position of x[..., 0, :],
        used by cached generation where T == 1."""
        dtype = x.dtype
        B, H, T, D = x.shape
        assert D == self.head_dim
        assert pos0 + T <= self.max_seq, f"position {pos0 + T} exceeds max_seq {self.max_seq}"

        half = D // 2
        x = x.float()
        x1, x2 = x[..., :half], x[..., half:]

        cos = self.cos[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)

        return torch.cat([x1 * cos - x2 * sin,
                          x2 * cos + x1 * sin], dim=-1).to(dtype)


class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq, rope=True):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"

        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.qkv      = nn.Linear(dim, 3 * dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope     = RoPE(self.head_dim, max_seq) if rope else None

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)   # (B, H, T, hd)

        if self.rope is not None:
            q, k = self.rope(q), self.rope(k)            # never v

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)


# ============================================================
# TESTS
# ============================================================

torch.manual_seed(42)
B, T, D, H = 2, 16, 768, 12

attn = CausalSelfAttention(D, H, max_seq=2048, rope=False)
x = torch.randn(B, T, D)
y = attn(x)
assert y.shape == x.shape and torch.isfinite(y).all()
print("attention  shape/finite ok")

# causality: perturbing position t must not affect positions < t
t = 7
x1 = torch.randn(B, T, D)
x2 = x1.clone()
x2[:, t, :] = torch.randn(B, D)
with torch.no_grad():
    y1, y2 = attn(x1), attn(x2)

assert torch.allclose(y1[:, :t], y2[:, :t], atol=1e-6), "LEAK: past positions changed"
assert not torch.allclose(y1[:, t:], y2[:, t:], atol=1e-6), "position t had no effect"
print("attention  causality ok")

# --- RoPE ---
hd, max_seq = 64, 2048
rope = RoPE(hd, max_seq)

q = torch.randn(2, 8, 32, hd)
assert torch.allclose(q.norm(dim=-1), rope(q).norm(dim=-1), atol=1e-5), "not a rotation"
print("rope       norm preservation ok")

# relative position: score depends only on (i - j), with DISTINCT q and k
qv, kv = torch.randn(hd), torch.randn(hd)
qt, kt = torch.zeros(1, 1, 16, hd), torch.zeros(1, 1, 16, hd)
qt[0, 0, 5], kt[0, 0, 3]   = qv, kv      # offset 2
qt[0, 0, 12], kt[0, 0, 10] = qv, kv      # offset 2
qr, kr = rope(qt), rope(kt)

d1 = (qr[0, 0, 5]  * kr[0, 0, 3]).sum()
d2 = (qr[0, 0, 12] * kr[0, 0, 10]).sum()
d3 = (qr[0, 0, 12] * kr[0, 0, 3]).sum()   # offset 9 — must differ

assert torch.allclose(d1, d2, atol=1e-5), f"not relative: {d1:.6f} vs {d2:.6f}"
assert not torch.allclose(d1, d3, atol=1e-3), "position has no effect at all"
print(f"rope       relative position ok  (off2 {d1:.4f}={d2:.4f}, off9 {d3:.4f})")

# pos0 offset must equal the same absolute position computed in one pass
full = rope(qt)
piece = rope(qt[:, :, 12:13], pos0=12)
assert torch.allclose(full[:, :, 12:13], piece, atol=1e-5), "pos0 offset broken"
print("rope       pos0 offset ok")

# --- combined ---
rattn = CausalSelfAttention(768, 12, max_seq=2048)
xr = torch.randn(2, 128, 768)
with torch.no_grad():
    yr = rattn(xr)
assert yr.shape == xr.shape and torch.isfinite(yr).all()
print(f"rope-attn  ok | params {sum(p.numel() for p in rattn.parameters()):,}")

In [ ]:
import math, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE = META["vocab_size"]
DIM, N_LAYERS, N_HEADS, MAX_SEQ = 768, 12, 12, 2048
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class TransformerBlock(nn.Module):
    def __init__(self, dim, n_heads, max_seq):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn  = CausalSelfAttention(dim, n_heads, max_seq)
        self.norm2 = RMSNorm(dim)
        self.mlp   = SwiGLU(dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class GPTModel(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, dim=DIM,
                 n_layers=N_LAYERS, n_heads=N_HEADS, max_seq=MAX_SEQ):
        super().__init__()
        self.vocab_size, self.dim = vocab_size, dim
        self.n_layers, self.n_heads, self.max_seq = n_layers, n_heads, max_seq

        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.blocks  = nn.ModuleList(
            [TransformerBlock(dim, n_heads, max_seq) for _ in range(n_layers)])
        self.norm_f  = RMSNorm(dim)
        self.head    = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight          # tie

        self.apply(self._init_weights)

        # rescale residual-writing projections AFTER the blanket init
        std = 0.02 / math.sqrt(2 * n_layers)
        for b in self.blocks:
            nn.init.normal_(b.attn.out_proj.weight, 0.0, std)
            nn.init.normal_(b.mlp.w3.weight,        0.0, std)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        T = idx.size(1)
        if T > self.max_seq:
            raise ValueError(f"T={T} exceeds max_seq={self.max_seq}")

        x = self.tok_emb(idx)
        for b in self.blocks:
            x = b(x)
        logits = self.head(self.norm_f(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size),
                                   targets.reshape(-1))
        return logits, loss


def count_params(m):
    """Dedupe by id() — tied head and embedding are one Parameter."""
    return sum(p.numel() for p in {id(p): p for p in m.parameters()}.values())


# ============================================================
torch.manual_seed(42)
model = GPTModel().to(DEVICE)

# --- 1. parameter count ---
total = count_params(model)
emb   = model.tok_emb.weight.numel()
blk   = sum(count_params(b) for b in model.blocks)

print(f"params {total:,} ({total/1e6:.2f}M)")
assert 50_000_000 <= total <= 300_000_000, f"{total:,} outside 50M-300M"
print(f"  embedding (tied) {emb:,}  ({100*emb/total:.1f}%)")
print(f"  blocks           {blk:,}  ({100*blk/total:.1f}%)")
print(f"  per block        {blk//N_LAYERS:,}")

# --- 2. initial loss ---
model.eval()
idx = torch.randint(0, VOCAB_SIZE, (2, 128), device=DEVICE)
tgt = torch.randint(0, VOCAB_SIZE, (2, 128), device=DEVICE)
with torch.no_grad():
    _, init_loss = model(idx, tgt)
init_loss = init_loss.item()

print(f"\ninitial loss {init_loss:.4f} vs ln(vocab) {math.log(VOCAB_SIZE):.4f}")
assert 9.0 < init_loss < 12.0, f"unexpected init loss {init_loss:.4f}"

# --- 3. end-to-end causality (relative criterion) ---
torch.manual_seed(123)
x1 = torch.randint(0, VOCAB_SIZE, (1, 64), device=DEVICE)
x2 = x1.clone()
t = 31
x2[0, t] = (x1[0, t].item() + 1) % VOCAB_SIZE

with torch.no_grad():
    y1, _ = model(x1)
    y2, _ = model(x2)

before = (y1[:, :t] - y2[:, :t]).abs().max().item()
after  = (y1[:, t:] - y2[:, t:]).abs().max().item()
print(f"\ncausality: max delta before t {before:.2e} | from t {after:.2e}")
assert before < after * 1e-4, "LEAK: future token affected past positions"
assert after > 1e-3, "perturbation had no effect"
print("causality ok")

# --- 4. overfit one real batch ---
train_mm = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
BS, SL = 2, 128
need = BS * (SL + 1)
chunk = torch.from_numpy(train_mm[:need].astype(np.int64)).to(DEVICE).view(BS, SL + 1)
bx, by = chunk[:, :-1], chunk[:, 1:]

torch.manual_seed(42)
om = GPTModel().to(DEVICE)
om.train()
opt = torch.optim.AdamW(om.parameters(), lr=1e-3)
amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp)

hist = []
for step in range(100):
    opt.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
        _, loss = om(bx, by)
    scaler.scale(loss).backward()
    scaler.step(opt); scaler.update()
    hist.append(loss.item())
    if step == 0 or (step + 1) % 20 == 0:
        print(f"  step {step+1:3d}/100  loss {loss.item():.4f}")

print(f"\noverfit {hist[0]:.4f} -> {hist[-1]:.4f}")
assert hist[-1] < 1.0, f"gradients not flowing: final loss {hist[-1]:.4f}"
print("overfit ok")

del om, opt, scaler
torch.cuda.empty_cache()
print(f"\nMODEL READY — {total/1e6:.2f}M params")

In [ ]:
import os, time, math, random
import numpy as np
import torch

BLOCK, MICRO_BS, ACCUM = 1024, 8, 4
TOKENS_PER_STEP = MICRO_BS * ACCUM * BLOCK

LR, MIN_LR, WARMUP, WD = 6e-4, 6e-5, 200, 0.1

MAX_STEPS = 6600           # ~4h at 15,006 tok/s -> 216M tokens, ~1 epoch
EVAL_EVERY, SAVE_EVERY, LOG_EVERY = 250, 250, 50
VAL_BATCHES, SEED = 20, 42

CKPT_PATH    = "/kaggle/working/ckpt/A_decoder_last.pt"
WEIGHTS_PATH = "/kaggle/working/ckpt/A_decoder.pt"
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_data   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")

batch_rng = np.random.RandomState(SEED + 1)


def get_batch(data, rng, bs):
    ix = rng.randint(0, len(data) - BLOCK - 1, size=bs)
    x = np.stack([data[i:i + BLOCK]         for i in ix]).astype(np.int64)
    y = np.stack([data[i + 1:i + 1 + BLOCK] for i in ix]).astype(np.int64)
    return (torch.from_numpy(x).to(DEVICE, non_blocking=True),
            torch.from_numpy(y).to(DEVICE, non_blocking=True))


def make_val_batches():
    """Fixed seed -> identical batches every eval and across architectures."""
    rng, out = np.random.RandomState(SEED + 2), []
    for _ in range(VAL_BATCHES):
        ix = rng.randint(0, len(val_data) - BLOCK - 1, size=MICRO_BS)
        x = np.stack([val_data[i:i + BLOCK]         for i in ix]).astype(np.int64)
        y = np.stack([val_data[i + 1:i + 1 + BLOCK] for i in ix]).astype(np.int64)
        out.append((torch.from_numpy(x), torch.from_numpy(y)))
    return out


val_batches = make_val_batches()

decay   = [p for p in model.parameters() if p.requires_grad and p.dim() >= 2]
nodecay = [p for p in model.parameters() if p.requires_grad and p.dim() < 2]

optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WD},
     {"params": nodecay, "weight_decay": 0.0}],
    lr=LR, betas=(0.9, 0.95), fused=(DEVICE == "cuda"))

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def get_lr(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    if MAX_STEPS <= WARMUP:
        return MIN_LR
    p = min(max((step - WARMUP) / (MAX_STEPS - WARMUP), 0.0), 1.0)
    return MIN_LR + (LR - MIN_LR) * 0.5 * (1.0 + math.cos(math.pi * p))


def save_checkpoint(step, hist):
    state = dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                 scaler=scaler.state_dict(), step=step, loss_history=hist,
                 batch_rng_state=batch_rng.get_state(),
                 torch_rng_state=torch.get_rng_state(),
                 cuda_rng_state=(torch.cuda.get_rng_state_all()
                                 if torch.cuda.is_available() else None),
                 config=dict(block=BLOCK, micro_bs=MICRO_BS, accum=ACCUM,
                             lr=LR, min_lr=MIN_LR, warmup=WARMUP,
                             wd=WD, max_steps=MAX_STEPS))
    torch.save(state, CKPT_PATH + ".tmp")
    os.replace(CKPT_PATH + ".tmp", CKPT_PATH)      # atomic

    fp16 = {k: v.detach().cpu().half() if v.is_floating_point() else v.detach().cpu()
            for k, v in model.state_dict().items()}
    torch.save(fp16, WEIGHTS_PATH + ".tmp")
    os.replace(WEIGHTS_PATH + ".tmp", WEIGHTS_PATH)


start_step, loss_history = 0, []

if os.path.exists(CKPT_PATH):
    ck = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    start_step   = ck["step"] + 1
    loss_history = ck.get("loss_history", [])
    batch_rng.set_state(ck["batch_rng_state"])
    torch.set_rng_state(ck["torch_rng_state"].cpu()
                        if torch.is_tensor(ck["torch_rng_state"]) else ck["torch_rng_state"])
    if torch.cuda.is_available() and ck.get("cuda_rng_state") is not None:
        torch.cuda.set_rng_state_all(ck["cuda_rng_state"])
    print(f"resumed from step {start_step}", flush=True)
    if start_step >= MAX_STEPS:
        print("already complete", flush=True)
else:
    print("starting fresh", flush=True)


@torch.no_grad()
def evaluate():
    model.eval()
    total = 0.0
    for xc, yc in val_batches:
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            _, l = model(xc.to(DEVICE), yc.to(DEVICE))
        total += l.item()
    model.train()
    return total / len(val_batches)


model.to(DEVICE); model.train()
print(f"device {DEVICE} | tokens/step {TOKENS_PER_STEP:,} | "
      f"train {len(train_data):,} tok | max_steps {MAX_STEPS} | "
      f"target {MAX_STEPS*TOKENS_PER_STEP/1e6:.0f}M tokens\n", flush=True)

warmup_done_t, warmup_done_step = None, start_step + 3
t_run = time.time()

for step in range(start_step, MAX_STEPS):
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    accum = torch.zeros((), device=DEVICE)

    for _ in range(ACCUM):
        x, y = get_batch(train_data, batch_rng, MICRO_BS)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            _, loss = model(x, y)
        scaler.scale(loss / ACCUM).backward()
        accum += loss.detach()                       # no sync inside the loop

    scaler.unscale_(optimizer)                       # unscale BEFORE clipping
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer); scaler.update()

    step_loss = (accum / ACCUM).item()
    loss_history.append(step_loss)

    if step == warmup_done_step:
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        warmup_done_t = time.time()

    if (step + 1) % LOG_EVERY == 0:
        if warmup_done_t and step > warmup_done_step:
            tps = ((step - warmup_done_step) * TOKENS_PER_STEP
                   / (time.time() - warmup_done_t))
            eta = (MAX_STEPS - step - 1) * TOKENS_PER_STEP / tps / 3600
            tps_s, eta_s = f"{tps:,.0f}", f"{eta:4.2f}h"
        else:
            tps_s, eta_s = "warmup", "  -  "
        print(f"step {step+1:6d}/{MAX_STEPS} | loss {step_loss:7.4f} | lr {lr:.2e} | "
              f"grad {float(gnorm):6.3f} | tok/s {tps_s} | "
              f"{(time.time()-t_run)/60:6.1f} min | eta {eta_s}", flush=True)

    if (step + 1) % EVAL_EVERY == 0:
        vl = evaluate()
        print(f"  >> val loss {vl:.4f} (ppl {math.exp(vl):.1f})", flush=True)

    if (step + 1) % SAVE_EVERY == 0 or step + 1 == MAX_STEPS:
        save_checkpoint(step, loss_history)
        print(f"  >> checkpoint @ step {step+1}", flush=True)

print(f"\nTRAINING COMPLETE — {step+1} steps, "
      f"{(step+1)*TOKENS_PER_STEP/1e6:.0f}M tokens, "
      f"{(time.time()-t_run)/3600:.2f} h", flush=True)
print(f"final train loss {step_loss:.4f} | final val loss {evaluate():.4f}", flush=True)